In [1]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import Python.simfun as sim

X1, y1, beta1, sim_info1 = sim.simfun1(
    n=180,
    p=100,
    seed=123,
    snr=2.5,
    true_prop=0.1,
    device=device,
    dtype=torch.float32,
)

rho_list = [0.0, 0.5, 0.8, 0.95]
X2, y2, beta2, sim_info2 = sim.simfun_block_corr(
    n=1000,
    p=100,
    seed=123,
    snr=2.5,
    true_prop=0.1,
    rho=0.8,
    block_size=10,
    device=device,
    dtype=torch.float32,
)

noise_x_list = [0.5, 0.25, 0.10, 0.05]
X3, y3, beta3, sim_info3 = sim.simfun_group_competition(
    n=1000,
    p=100,
    seed=123,
    snr=2.5,
    true_prop=0.1,
    group_size=10,
    noise_x=0.15,
    one_active_per_group=True,
    device=device,
    dtype=torch.float32,
)

In [ ]:
%load_ext autoreload
%autoreload 2

import Python.benchmark as bm
import Python.config as cfg
import Python.framework as fw
import Python.model as md

schedule_cfg = cfg.StagewiseAnnealConfig(
    tau_start=0.5,
    tau_end=0.1,
    warmup_epochs=300,
    n_anneal_stages=20,
    min_stage_epochs=100,
    max_stage_epochs=400,
    base_lr=5e-5,
    stage_lr_decay=0.7,
    eval_every=25,
    print_every=100,
    diag_R_train=256,
    diag_R_final=4000,
    support_threshold=0.5,
)

mf_sas_cfg = bm.MFSpikeSlabConfig(
    pi=0.1,
    slab_var=20.0,
    max_iter=1000,
    tol=1e-6,
    update_sigma2=True,
    support_threshold=0.5,
    standardize_x=True,
    center_y=True,
)

mf_ard_cfg = bm.MFARDConfig(
    a0=1e-2,
    b0=1e-2,
    c0=1e-2,
    d0=1e-2,
    min_sigma2=1e-8,
    max_iter=1000,
    tol=1e-6,
    beta_eps=0.1,
    support_threshold=0.5,
    standardize_x=True,
    center_y=True,
)

lasso_cfg = bm.MFBayesLassoConfig(
    max_iter=1000,
    tol=1e-6,
    beta_eps=0.1,
    support_threshold=0.5,
    standardize_x=True,
    center_y=True,
)

method_cfgs = {
    "mf_spike_slab": mf_sas_cfg,
    "mf_ard": mf_ard_cfg,
    "mf_bayes_lasso": lasso_cfg,
}

split_cfg = cfg.SplitConfig(
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=123,
)

save_mf = cfg.SaveConfig(
    output_dir="./data/meanfield_seed123",
    save_summary_csv=True,
    save_results_pickle=True,
    save_metadata_json=True,
    save_manifest_json=True,
    save_history_csv=True,
    save_predictions_csv=True,
    save_var_table_csv=True,
    save_support_sets_json=True,
    save_yhat_csv=False,
)

save_flow = cfg.SaveConfig(
    output_dir="./data/flow_seed123",
    save_summary_csv=False,
    save_results_pickle=True,
    save_metadata_json=True,
    save_manifest_json=True,
    save_history_csv=True,
    save_predictions_csv=True,
    save_var_table_csv=True,
    save_support_sets_json=True,
    save_yhat_csv=False,
)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
out1 = fw.simflow_stagewise(
    X=X1,
    y=y1,
    beta_true=beta1,
    sim_info=sim_info1,
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    seed=123,
    K_q=32,
    K_g=16,
    schedule_cfg=schedule_cfg,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
{'n': 180, 'p': 100, 'n_active': 10, 'sigma2': 3.9107933044433594, 'sigma': 1.9775725788054808, 'active_idx': array([14, 17, 20, 32, 35, 43, 45, 62, 66, 95], dtype=int64), 'snr': 2.5}
[stage 00 | epoch 0001] tau=0.5000 lr=5.00e-05 loss=740.517334 logg=6.1323 dlogg=+0.0000 S= 12 churn=0.000 ok*
[stage 00 | epoch 0100] tau=0.5000 lr=5.00e-05 loss=315.506134 logg=5.1222 dlogg=+0.0039 S=  8 churn=0.750 ok*
[stage 00 | epoch 0200] tau=0.5000 lr=5.00e-05 loss=201.346603 logg=5.9017 dlogg=-1.3157 S= 10 churn=0.100 ok
[stage 00 | epoch 0300] tau=0.5000 lr=5.00e-05 loss=184.737442 logg=5.3228 dlogg=+0.1646 S=  9 churn=0.000 ok
[stage 01 | epoch 0301] tau=0.4613 lr=5.00e-05 loss=185.322357 logg=5.3809 dlogg=+0.0581 S=  9 churn=0.000 ok*
[stage 01 | epoch 0400] tau=0.4613 lr=5.00e-05 loss=180.176666 logg=4.8139 dlogg=-0.1151 S=  8 churn=0.000 ok*
[stage 01 | epoch 0500] tau=0.4613 lr=5.00e-05 loss=179.053680 logg=4.6652 dlogg=-0.0614 S=  8 churn=0.000 ok
[stage 01 | ep

In [53]:
mfresults1, mftable1 = bm.run_benchmark(
    X=X1,
    y=y1,
    beta_true=beta1,
    active_idx=sim_info1["active_idx"],
    seed=123,
    sim_info=sim_info1,
    methods=[
        "mf_spike_slab",
        "mf_ard",
        "mf_bayes_lasso",
    ],
    split_cfg=split_cfg,
    method_cfgs=method_cfgs,
)

for out in mfresults1:
    bm.print_result(out, top_k=20)
    print("\n" + "=" * 80 + "\n")

===== mf_spike_slab result =====
seed          : 123
runtime_sec   : 0.019682
converged     : True
n_iter        : 22
threshold     : 0.5
sigma2        : 4.375956037033296
selected_size : 4

===== Selection metrics =====
 precision  recall       f1  fdr  tp  fp  fn   tn  support_size    auroc    auprc
       1.0     0.4 0.571429  0.0 4.0 0.0 6.0 90.0           4.0 0.964444 0.865309

===== Selected support =====
[14, 17, 35, 62]

===== Top 20 variables =====
 j  beta_mean  beta_sd  support_score  selected  beta_true  truth  abs_beta_mean
14  -1.638897 0.203211       1.000000         1  -1.628044      1       1.638897
17  -1.552172 0.201396       1.000000         1  -1.645482      1       1.552172
35  -1.243792 0.208458       0.999996         1  -1.140157      1       1.243792
62  -1.086923 0.223224       0.998960         1  -1.142017      1       1.086923
66   0.331031 0.371655       0.484611         0   0.751040      1       0.331031
95   0.122229 0.247550       0.220296         0   0.

In [ ]:
out2 = fw.simflow_stagewise(
    X=X2,
    y=y2,
    beta_true=beta2,
    sim_info=sim_info2,
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    seed=123,
    K_q=32,
    K_g=16,
    schedule_cfg=schedule_cfg,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
{'sim': 'block_corr', 'n': 1000, 'p': 100, 'n_active': 10, 'active_idx': array([ 3, 23, 30, 31, 34, 46, 55, 69, 81, 94], dtype=int64), 'snr': 2.5, 'sigma2': 7.5747757380663305, 'sigma': 2.7522310473625446, 'rho': 0.8, 'block_size': 10, 'beta_low': 0.8, 'beta_high': 1.8}
[stage 00 | epoch 0001] tau=0.5000 lr=5.00e-05 loss=2617.333984 logg=7.8239 dlogg=+0.0000 S= 20 churn=0.000 ok*
[stage 00 | epoch 0100] tau=0.5000 lr=5.00e-05 loss=1616.512695 logg=5.4395 dlogg=-0.4875 S= 11 churn=0.676 ok*
[stage 00 | epoch 0200] tau=0.5000 lr=5.00e-05 loss=1450.812378 logg=5.9678 dlogg=+0.5766 S= 10 churn=0.000 ok
[stage 00 | epoch 0300] tau=0.5000 lr=5.00e-05 loss=1426.172974 logg=6.1339 dlogg=+0.5998 S= 10 churn=0.000 ok
[stage 01 | epoch 0301] tau=0.4613 lr=5.00e-05 loss=1428.246704 logg=6.2358 dlogg=+0.1019 S= 10 churn=0.000 ok*
[stage 01 | epoch 0400] tau=0.4613 lr=5.00e-05 loss=1418.466797 logg=5.8791 dlogg=-0.2440 S= 10 churn=0.000 ok
[stage 01 | epoch 0500] tau=0.46

In [52]:
mfresults2, mftable2 = bm.run_benchmark(
    X=X2,
    y=y2,
    beta_true=beta2,
    active_idx=sim_info2["active_idx"],
    seed=123,
    sim_info=sim_info2,
    methods=[
        "mf_spike_slab",
        "mf_ard",
        "mf_bayes_lasso",
    ],
    split_cfg=split_cfg,
    method_cfgs=method_cfgs,
)

for out in mfresults2:
    bm.print_result(out, top_k=20)
    print("\n" + "=" * 80 + "\n")

===== mf_spike_slab result =====
seed          : 123
runtime_sec   : 0.036458
converged     : True
n_iter        : 36
threshold     : 0.5
sigma2        : 7.309169241276295
selected_size : 10

===== Selection metrics =====
 precision  recall  f1  fdr   tp  fp  fn   tn  support_size  auroc  auprc
       1.0     1.0 1.0  0.0 10.0 0.0 0.0 90.0          10.0    1.0    1.0

===== Selected support =====
[3, 23, 30, 31, 34, 46, 55, 69, 81, 94]

===== Top 20 variables =====
 j  beta_mean  beta_sd  support_score  selected  beta_true  truth  abs_beta_mean
81  -1.862684 0.109567       1.000000         1  -1.742517      1       1.862684
30  -1.597860 0.110489       1.000000         1  -1.586958      1       1.597860
46  -1.595759 0.108426       1.000000         1  -1.611022      1       1.595759
 3   1.511227 0.112762       1.000000         1   1.590818      1       1.511227
34   1.503517 0.109397       1.000000         1   1.486087      1       1.503517
31  -1.189458 0.110703       1.000000       

In [ ]:
out3 = fw.simflow_stagewise(
    X=X3,
    y=y3,
    beta_true=beta3,
    sim_info=sim_info3,
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    seed=123,
    K_q=16,
    K_g=8,
    schedule_cfg=schedule_cfg,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
{'sim': 'group_competition', 'n': 1000, 'p': 100, 'n_active': 10, 'active_idx': array([ 0, 17, 20, 35, 42, 56, 66, 75, 85, 99]), 'snr': 2.5, 'sigma2': 6.648223662989551, 'sigma': 2.578414951668864, 'group_size': 10, 'n_groups': 10, 'noise_x': 0.15, 'one_active_per_group': True, 'group_id': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8,
       8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]), 'active_groups': array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), 'beta_low': 0.8, 'beta_high': 1.8}
[stage 00 | epoch 0001] tau=0.5000 lr=5.00e-05 loss=2586.267822 logg=8.0564 dlogg=+0.0000 S= 50 churn=0.000 ok*
[stage 00 | epoch 0100] tau=0.5000 lr=5.00e-05 loss=1757.869629 logg=5.3160 dlogg=-0.3121 S= 56 churn=0.378 ok
[stage 00 | epoch 0200] tau=0.5000 lr=5.00e-

In [ ]:
out4 = fw.simflow_stagewise(
    X=X3,
    y=y3,
    beta_true=beta3,
    sim_info=sim_info3,
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    seed=123,
    K_q=32,
    K_g=8,
    schedule_cfg=schedule_cfg,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
{'sim': 'group_competition', 'n': 1000, 'p': 100, 'n_active': 10, 'active_idx': array([ 0, 17, 20, 35, 42, 56, 66, 75, 85, 99]), 'snr': 2.5, 'sigma2': 6.648223662989551, 'sigma': 2.578414951668864, 'group_size': 10, 'n_groups': 10, 'noise_x': 0.15, 'one_active_per_group': True, 'group_id': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8,
       8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]), 'active_groups': array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), 'beta_low': 0.8, 'beta_high': 1.8}
[stage 00 | epoch 0001] tau=0.5000 lr=5.00e-05 loss=2585.614502 logg=8.3077 dlogg=+0.0000 S= 40 churn=0.000 ok*
[stage 00 | epoch 0100] tau=0.5000 lr=5.00e-05 loss=1647.772339 logg=5.4968 dlogg=-0.1281 S=  7 churn=0.806 ok*
[stage 00 | epoch 0200] tau=0.5000 lr=5.00e

In [54]:
mfresults3, mftable3 = bm.run_benchmark(
    X=X3,
    y=y3,
    beta_true=beta3,
    active_idx=sim_info3["active_idx"],
    seed=123,
    sim_info=sim_info3,
    methods=[
        "mf_spike_slab",
        "mf_ard",
        "mf_bayes_lasso",
    ],
    split_cfg=split_cfg,
    method_cfgs=method_cfgs,
)

for out in mfresults3:
    bm.print_result(out, top_k=20)
    print("\n" + "=" * 80 + "\n")

===== mf_spike_slab result =====
seed          : 123
runtime_sec   : 0.006760
converged     : True
n_iter        : 6
threshold     : 0.5
sigma2        : 7.3128125097356875
selected_size : 10

===== Selection metrics =====
 precision  recall  f1  fdr  tp  fp  fn   tn  support_size    auroc  auprc
       0.2     0.2 0.2  0.8 2.0 8.0 8.0 82.0          10.0 0.742222 0.2699

===== Selected support =====
[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]

===== Top 20 variables =====
 j  beta_mean  beta_sd  support_score  selected  beta_true  truth  abs_beta_mean
 0   1.772306 0.111522       1.000000         1   1.711361      1       1.772306
20  -1.700128 0.112344       1.000000         1  -1.668201      1       1.700128
50  -1.415176 0.110240       1.000000         1   0.000000      0       1.415176
60  -1.339247 0.111117       1.000000         1   0.000000      0       1.339247
10   1.195504 0.111086       1.000000         1   0.000000      0       1.195504
90  -1.000120 0.109996       1.000000     